<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/email-management-api/blob/main/qwen25_email_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [11]:
!pip install -q pandas openpyxl

In [12]:
import pandas as pd
import json
import ast

Get Excel Sheet

In [13]:
from google.colab import files
uploaded = files.upload()
excel_file = next(iter(uploaded.keys()))
print("Uploaded:", excel_file)

Saving manual_classification_updated.xlsx to manual_classification_updated (1).xlsx
Uploaded: manual_classification_updated (1).xlsx


Extract the data from sheet

In [14]:
df = pd.read_excel(excel_file)

required_columns = [
    "subject",
    "from",
    "body",
    "sender_type",
    "is_action_required",
    "labels_to_remove",
    "labels_to_add"
]

df = df[required_columns]

print("Total records:", len(df))
print(df.head())

Total records: 1981
                                             subject  \
0  RE: Please urgently investigate and prevent de...   
1  Learn more about MongoDB 9.0 - the best versio...   
2  AI Agents (07): Agentic Harness (Frameworks) |...   
3  SN 59116 Action recommended: Upgrade your App ...   
4  Fwd: TL Team Branch / Collection officer and R...   

                                                from  \
0          "Sudhakar.K" <sudhakar.k@arisecapital.in>   
1         MongoDB <mongodbteam@messages.mongodb.com>   
2           Medium Daily Digest <noreply@medium.com>   
3      Microsoft Azure <azure-noreply@microsoft.com>   
4  JothishKumar SP <jothishkumar.sp@arisecapital.in>   

                                                body sender_type  \
0  Hi Team,\n\n\n\n \n\n\n\nWe would like to unde...    internal   
1  Webinar\n\nMongoDB 9.0: Trusted at Scale, Buil...    external   
2  Stories for Prasannavenkatesh M\n\n@prasannave...    external   
3  <!-- SUBJECT: Action recommende

In [15]:
def parse_labels(value):

    if isinstance(value, list):
        return value

    if pd.isna(value) or str(value).strip() == "":
        return []

    return ast.literal_eval(value)


def convert_record(row):

    input_data = {
        "subject": row["subject"],
        "from": row["from"],
        "body": row["body"],
        "sender_type": row["sender_type"],
        "is_action_required": row["is_action_required"]
    }

    output_data = {
        "labels_to_remove": parse_labels(row["labels_to_remove"]),
        "labels_to_add": parse_labels(row["labels_to_add"])
    }

    return {
        "input": json.dumps(input_data, ensure_ascii=False),
        "output": json.dumps(output_data, ensure_ascii=False)
    }


dataset = [
    convert_record(row)
    for _, row in df.iterrows()
]

print("Conversion completed!")
print("Total records:", len(dataset))
print(json.dumps(dataset[0], indent=4, ensure_ascii=False))

Conversion completed!
Total records: 1981
{
    "input": "{\"subject\": \"RE: Please urgently investigate and prevent deletion of six expired Azure subscriptions on 19 November 2026,ensuring ... - TrackingID#2611160030004891\", \"from\": \"\\\"Sudhakar.K\\\" <sudhakar.k@arisecapital.in>\", \"body\": \"Hi Team,\\n\\n\\n\\n \\n\\n\\n\\nWe would like to understand why the issue was identified as an Azure billing matter only after reviewing our case details, despite having already raised two tickets for the same issue.\\n\\n\\n\\nExisting Support IDs:\\n\\n\\n\\n*\\t2611160030004891\\n\\n*\\t2611120030004271\\n\\n\\n\\nNeither ticket provided clear instructions on how to reach the appropriate Azure Billing Support team. Instead, after a day's wait, we were advised to create another ticket.\\n\\n\\n\\nConsidering the urgency, we request you to please create or escalate a support request on our behalf to the appropriate Azure Billing Support team, rather than asking us to repeat the process.